# CFLOW Full Two-Stage (Severstal, Colab)

This notebook runs a strict two-stage pipeline:
1. Stage 1 = official CFLOW-AD script path (`main.py --action-type norm-train/norm-test`)
2. Stage 2 = ResNet50 known-class classifier
3. Integrated decisions: `no defect` / `known class` / `unknown defect`

No proxy fallback is used for stage-1.


In [1]:
"""# 0) Optional one-time fix for NumPy ABI (run, then restart runtime)
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy==1.26.4'])
print('Installed numpy==1.26.4. Restart runtime now, then run from cell 1.')
"""

"# 0) Optional one-time fix for NumPy ABI (run, then restart runtime)\nimport sys, subprocess\nsubprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'numpy==1.26.4'])\nprint('Installed numpy==1.26.4. Restart runtime now, then run from cell 1.')\n"

In [2]:
# 1) Runtime preflight
import sys, numpy as np, torch, torchvision
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('TorchVision:', torchvision.__version__)
assert np.__version__.startswith('1.26'), 'Need numpy==1.26.x'
print('Preflight OK')


Python: 3.12.12
NumPy: 1.26.4
Torch: 2.10.0+cpu
TorchVision: 0.25.0+cpu
Preflight OK


In [3]:
# 2) Repo setup (FYP + CFLOW) + deps
import os, sys, subprocess
from pathlib import Path

FYP_REPO = Path('/content/FYP-code')
if not FYP_REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(FYP_REPO)])
subprocess.check_call(['git', '-C', str(FYP_REPO), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(FYP_REPO), 'reset', '--hard', 'origin/main'])

CFLOW_REPO = Path('/content/cflow-ad')
if not CFLOW_REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/gudovskiy/cflow-ad.git', str(CFLOW_REPO)])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.7', 'FrEIA==0.2', 'scikit-learn==1.3.2'])

os.chdir(FYP_REPO)
for p in [str(FYP_REPO), str(FYP_REPO/'src'), str(FYP_REPO/'severstral-osr'/'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('FYP repo:', FYP_REPO)
print('CFLOW repo:', CFLOW_REPO)


FYP repo: /content/FYP-code
CFLOW repo: /content/cflow-ad


In [4]:
# 3) Drive + dataset checks
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DATA_ROOT = Path('/content/drive/MyDrive/datasets/severstal')
TRAIN_CSV = DATA_ROOT / 'train.csv'
TRAIN_IMAGES = DATA_ROOT / 'train_images'
OUT_ROOT = Path('/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.exists(), f'Missing: {TRAIN_CSV}'
assert TRAIN_IMAGES.exists() and TRAIN_IMAGES.is_dir(), f'Missing: {TRAIN_IMAGES}'

print('DATA_ROOT:', DATA_ROOT)
print('OUT_ROOT :', OUT_ROOT)


Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/datasets/severstal
OUT_ROOT : /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full


In [5]:
# 4) Build or reuse split manifest
import csv, json, random
from collections import defaultdict

SPLIT_PATH = OUT_ROOT / 'split_manifest.json'
FORCE_REBUILD_SPLIT = False

if SPLIT_PATH.exists() and not FORCE_REBUILD_SPLIT:
    split = json.loads(SPLIT_PATH.read_text())
    print('Reusing split:', SPLIT_PATH)
else:
    SEED = 42
    KNOWN_CLASSES = ['Class_1', 'Class_2', 'Class_3']
    rng = random.Random(SEED)

    rows_by_image = defaultdict(list)
    with open(TRAIN_CSV, 'r', newline='') as f:
        rd = csv.DictReader(f)
        for r in rd:
            rows_by_image[r['ImageId'].strip()].append(r)

    all_images = [p for p in TRAIN_IMAGES.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}]
    normal_paths, single_label = [], []

    for p in all_images:
        rows = rows_by_image.get(p.name, [])
        pos = set()
        for r in rows:
            enc = str(r.get('EncodedPixels','')).strip().lower()
            if enc and enc != 'nan':
                pos.add(int(r['ClassId']))
        if len(pos)==0:
            normal_paths.append(str(p))
        elif len(pos)==1:
            single_label.append((str(p), f"Class_{next(iter(pos))}"))

    known_all = [(p,c) for p,c in single_label if c in KNOWN_CLASSES]
    unknown_all = [(p,c) for p,c in single_label if c not in KNOWN_CLASSES]

    def stratified_split(samples, train_ratio=0.7, val_ratio=0.15, seed=42):
        rr = random.Random(seed)
        by=defaultdict(list)
        for s in samples: by[s[1]].append(s)
        tr,va,te=[],[],[]
        for items in by.values():
            rr.shuffle(items)
            n=len(items); ntr=int(n*train_ratio); nva=int(n*val_ratio)
            tr += items[:ntr]; va += items[ntr:ntr+nva]; te += items[ntr+nva:]
        return tr,va,te

    known_train, known_val, known_test = stratified_split(known_all, seed=SEED)
    rng.shuffle(normal_paths); rng.shuffle(unknown_all)

    split = {
        'seed': SEED,
        'known_classes': KNOWN_CLASSES,
        'normal_train': normal_paths[:1200],
        'normal_val': normal_paths[1200:1500],
        'normal_test': normal_paths[1500:1800],
        'known_train': known_train[:900],
        'known_val': known_val[:300],
        'known_test': known_test[:300],
        'unknown_test': unknown_all[:300],
    }

    if min(len(split['normal_train']),len(split['normal_val']),len(split['normal_test']),len(split['known_train']),len(split['known_val']),len(split['known_test']),len(split['unknown_test'])) == 0:
        raise RuntimeError('Split contains empty groups.')

    SPLIT_PATH.write_text(json.dumps(split, indent=2))
    print('Built split:', SPLIT_PATH)

print({k: len(v) for k,v in split.items() if isinstance(v,list)})


Built split: /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/split_manifest.json
{'known_classes': 3, 'normal_train': 1200, 'normal_val': 300, 'normal_test': 300, 'known_train': 900, 'known_val': 300, 'known_test': 300, 'unknown_test': 300}


In [6]:
# 5) Build CFLOW dataset directories (MVTec-AD style) from split
import os, shutil, json
from pathlib import Path

split = json.loads((OUT_ROOT / 'split_manifest.json').read_text())

# This CFLOW repo expects ./data/MVTec-AD/<class>/...
CFLOW_CLASS = 'bottle'
CFLOW_VAL_CLASS = 'bottle_valonly'
CFLOW_DATA = Path('/content/cflow-ad/data/MVTec-AD') / CFLOW_CLASS
CFLOW_VAL_DATA = Path('/content/cflow-ad/data/MVTec-AD') / CFLOW_VAL_CLASS

for root in [CFLOW_DATA, CFLOW_VAL_DATA]:
    if root.exists():
        shutil.rmtree(root)

for p in [
    CFLOW_DATA/'train'/'good',
    CFLOW_DATA/'test'/'good',
    CFLOW_DATA/'test'/'known_defect',
    CFLOW_DATA/'test'/'unknown_defect',
    CFLOW_VAL_DATA/'train'/'good',
    CFLOW_VAL_DATA/'test'/'good',
]:
    p.mkdir(parents=True, exist_ok=True)

def link_many(rows, dst, labeled=False):
    for i, row in enumerate(rows):
        src = Path(row[0] if labeled else row)
        if not src.exists():
            continue
        out = dst / f'{i:06d}_{src.name}'
        try:
            os.symlink(src, out)
        except Exception:
            shutil.copy2(src, out)

# main eval dataset
link_many(split['normal_train'], CFLOW_DATA/'train'/'good')
link_many(split['normal_test'], CFLOW_DATA/'test'/'good')
link_many(split['known_test'], CFLOW_DATA/'test'/'known_defect', labeled=True)
link_many(split['unknown_test'], CFLOW_DATA/'test'/'unknown_defect', labeled=True)

# val-only dataset for tau calibration
link_many(split['normal_train'], CFLOW_VAL_DATA/'train'/'good')
link_many(split['normal_val'], CFLOW_VAL_DATA/'test'/'good')

(OUT_ROOT/'cflow_paths.json').write_text(json.dumps({
    'class_main': CFLOW_CLASS,
    'class_val': CFLOW_VAL_CLASS,
    'main_root': str(CFLOW_DATA),
    'val_root': str(CFLOW_VAL_DATA),
}, indent=2))

print('Built CFLOW dataset roots:')
print('-', CFLOW_DATA)
print('-', CFLOW_VAL_DATA)


Built CFLOW dataset roots:
- /content/cflow-ad/data/MVTec/bottle
- /content/cflow-ad/data/MVTec/bottle_valonly


In [6]:
# 6) Run official CFLOW quick sanity (tiny epochs) on main class
import subprocess, time, json

cfg = json.loads((OUT_ROOT/'cflow_paths.json').read_text())
cwd = '/content/cflow-ad'
CLASS_MAIN = cfg['class_main']
RUN_ID = 1  # must be int for this repo

cmd_train = [
    'python', 'main.py',
    '--dataset', 'mvtec',
    '--class-name', CLASS_MAIN,
    '--action-type', 'norm-train',
    '--run-name', str(RUN_ID),
    '--batch-size', '8',
    '--meta-epochs', '1',
    '--sub-epochs', '1',
    '--input-size', '256',
    '--gpu', '0',
]
cmd_test = [
    'python', 'main.py',
    '--dataset', 'mvtec',
    '--class-name', CLASS_MAIN,
    '--action-type', 'norm-test',
    '--run-name', str(RUN_ID),
    '--batch-size', '8',
    '--input-size', '256',
    '--gpu', '0',
]

start = time.time()
r_train = subprocess.run(cmd_train, cwd=cwd, capture_output=True, text=True)
print(r_train.stdout[-3000:])
print(r_train.stderr[-3000:])
if r_train.returncode != 0:
    raise RuntimeError(f'CFLOW quick train failed with code {r_train.returncode}')

r_test = subprocess.run(cmd_test, cwd=cwd, capture_output=True, text=True)
print(r_test.stdout[-3000:])
print(r_test.stderr[-3000:])
if r_test.returncode != 0:
    raise RuntimeError(f'CFLOW quick test failed with code {r_test.returncode}')

print('CFLOW quick sanity done in %.1fs' % (time.time()-start))


LR schedule: [0, 0, 0]
Number of pool layers = 3
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth
CNF coder: 512
CNF coder: 1024
CNF coder: 2048


  0%|          | 0.00/132M [00:00<?, ?B/s]
 15%|█▌        | 20.2M/132M [00:00<00:00, 211MB/s]
 38%|███▊      | 49.9M/132M [00:00<00:00, 269MB/s]
 64%|██████▍   | 84.5M/132M [00:00<00:00, 311MB/s]
 94%|█████████▍| 124M/132M [00:00<00:00, 350MB/s] 
100%|██████████| 132M/132M [00:00<00:00, 323MB/s]
/usr/local/lib/python3.12/dist-packages/FrEIA/modules/all_in_one_block.py:119: UserWarning: Soft permutation will take a very long time to initialize with 1024 feature channels. Consider using hard permutation instead.
  warnings.warn(("Soft permutation will take a very long time to initialize "
/usr/local/lib/python3.12/dist-packages/FrEIA/modules/all_in_one_block.py:119: UserWarning: Soft permutation will take a very long time to initialize with 2048 f

RuntimeError: CFLOW quick train failed with code 1

In [ ]:
# 7) Run official CFLOW full stage-1 (main + val-only test)
import subprocess, json, time, os, shutil
from pathlib import Path

cfg = json.loads((OUT_ROOT/'cflow_paths.json').read_text())
cwd = '/content/cflow-ad'
CLASS_MAIN = cfg['class_main']
CLASS_VAL = cfg['class_val']
RUN_ID = 2  # integer run id

start = time.time()

# Train on main class dataset
subprocess.check_call([
    'python', 'main.py',
    '--dataset', 'mvtec',
    '--class-name', CLASS_MAIN,
    '--action-type', 'norm-train',
    '--run-name', str(RUN_ID),
    '--batch-size', '16',
    '--meta-epochs', '8',
    '--sub-epochs', '4',
    '--input-size', '256',
    '--gpu', '0',
], cwd=cwd)

# Test on main class dataset (normal_test + known/unknown defects)
subprocess.check_call([
    'python', 'main.py',
    '--dataset', 'mvtec',
    '--class-name', CLASS_MAIN,
    '--action-type', 'norm-test',
    '--run-name', str(RUN_ID),
    '--batch-size', '16',
    '--input-size', '256',
    '--gpu', '0',
], cwd=cwd)
print('CFLOW full main test done in %.1fs' % (time.time()-start))

# Val-only test for tau calibration.
# Use directory swap so model/checkpoint identity (class_main + run_id) stays consistent.
base = Path('/content/cflow-ad/data/MVTec-AD')
main_root = base / CLASS_MAIN
val_root = base / CLASS_VAL
backup = base / f'{CLASS_MAIN}_backup_for_val'

if backup.exists():
    shutil.rmtree(backup)
os.rename(main_root, backup)
os.rename(val_root, main_root)

try:
    subprocess.check_call([
        'python', 'main.py',
        '--dataset', 'mvtec',
        '--class-name', CLASS_MAIN,
        '--action-type', 'norm-test',
        '--run-name', str(RUN_ID),
        '--batch-size', '16',
        '--input-size', '256',
        '--gpu', '0',
    ], cwd=cwd)
finally:
    os.rename(main_root, val_root)
    os.rename(backup, main_root)

print('CFLOW val-only test done.')


In [ ]:
# 8) Extract CFLOW score vectors from official outputs (hard-fail if not found)
import os, json, numpy as np, pandas as pd, pickle
from pathlib import Path

split = json.loads((OUT_ROOT/'split_manifest.json').read_text())
n_main = len(split['normal_test']) + len(split['known_test']) + len(split['unknown_test'])
n_val = len(split['normal_val'])

base = Path('/content/cflow-ad')

# Find recently modified candidate files with score-like content
cands=[]
for p in base.rglob('*'):
    if not p.is_file():
        continue
    if p.suffix.lower() not in {'.npy','.npz','.pkl','.csv','.txt'}:
        continue
    name = p.name.lower()
    if any(k in name for k in ['score','scores','anom','prob','logp','result','infer']):
        cands.append(p)

cands = sorted(cands, key=lambda x: x.stat().st_mtime, reverse=True)

def read_vector(p):
    try:
        if p.suffix == '.npy':
            a=np.load(p, allow_pickle=True)
            if isinstance(a, np.ndarray):
                a=np.asarray(a).reshape(-1)
                if np.issubdtype(a.dtype, np.number): return a.astype(float)
        if p.suffix == '.npz':
            z=np.load(p, allow_pickle=True)
            for k in z.files:
                a=np.asarray(z[k]).reshape(-1)
                if np.issubdtype(a.dtype, np.number):
                    return a.astype(float)
        if p.suffix == '.pkl':
            with open(p,'rb') as f: obj=pickle.load(f)
            if isinstance(obj, (list,tuple,np.ndarray)):
                a=np.asarray(obj).reshape(-1)
                if np.issubdtype(a.dtype, np.number): return a.astype(float)
            if isinstance(obj, dict):
                for v in obj.values():
                    if isinstance(v,(list,tuple,np.ndarray)):
                        a=np.asarray(v).reshape(-1)
                        if np.issubdtype(a.dtype, np.number): return a.astype(float)
        if p.suffix == '.csv':
            df=pd.read_csv(p)
            for col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    a=df[col].to_numpy().reshape(-1)
                    return a.astype(float)
    except Exception:
        return None
    return None

main_scores = None
val_scores = None
picked=[]
for p in cands:
    vec=read_vector(p)
    if vec is None:
        continue
    if main_scores is None and len(vec)==n_main:
        main_scores=vec
        picked.append(('main',str(p),len(vec)))
    elif val_scores is None and len(vec)==n_val:
        val_scores=vec
        picked.append(('val',str(p),len(vec)))
    if main_scores is not None and val_scores is not None:
        break

print('Picked artifacts:', picked)
if main_scores is None:
    raise RuntimeError(f'Could not find main CFLOW score vector with length {n_main}. Inspect /content/cflow-ad outputs.')
if val_scores is None:
    raise RuntimeError(f'Could not find val CFLOW score vector with length {n_val}. Inspect /content/cflow-ad outputs.')

# Assume ImageFolder-like ordering: good, known_defect, unknown_defect
n_g = len(split['normal_test']); n_k = len(split['known_test']); n_u = len(split['unknown_test'])
if len(main_scores) != n_g+n_k+n_u:
    raise RuntimeError('Main scores length mismatch with split counts.')

s_nts = main_scores[:n_g]
s_kts = main_scores[n_g:n_g+n_k]
s_uts = main_scores[n_g+n_k:]
s_nva = val_scores

np.save(OUT_ROOT/'stage1_scores_normal_val.npy', s_nva)
np.save(OUT_ROOT/'stage1_scores_normal_test.npy', s_nts)
np.save(OUT_ROOT/'stage1_scores_known_test.npy', s_kts)
np.save(OUT_ROOT/'stage1_scores_unknown_test.npy', s_uts)

print('Saved stage1 score files to OUT_ROOT')


In [ ]:
# 9) Stage 2: known-class ResNet50 classifier
import json, time, numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from src.models.resnet50 import build_resnet50

split = json.loads((OUT_ROOT/'split_manifest.json').read_text())
known_classes = split['known_classes']
class_to_idx = {c:i for i,c in enumerate(known_classes)}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class KnownDS(Dataset):
    def __init__(self, rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        p,c=self.rows[i]
        return tf(Image.open(p).convert('RGB')), class_to_idx[c]

tr_dl=DataLoader(KnownDS(split['known_train']), batch_size=32, shuffle=True, num_workers=2)
va_dl=DataLoader(KnownDS(split['known_val']), batch_size=32, shuffle=False, num_workers=2)

model=build_resnet50(num_classes=len(known_classes), pretrained=True).to(device)
opt=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit=nn.CrossEntropyLoss()

best_va=-1.0; best=None
for ep in range(4):
    model.train()
    for x,y in tr_dl:
        x,y=x.to(device),y.to(device)
        opt.zero_grad(); loss=crit(model(x),y); loss.backward(); opt.step()
    model.eval(); cor=tot=0
    with torch.no_grad():
        for x,y in va_dl:
            x,y=x.to(device),y.to(device)
            pr=model(x).argmax(1)
            cor += (pr==y).sum().item(); tot += len(y)
    va=cor/max(1,tot)
    print(f'epoch {ep+1}/4 val_acc={va:.4f}')
    if va>best_va:
        best_va=va; best={k:v.cpu() for k,v in model.state_dict().items()}

model.load_state_dict(best)

def logits_for(rows):
    class DS(Dataset):
        def __init__(self, rows): self.rows=rows
        def __len__(self): return len(self.rows)
        def __getitem__(self, i):
            r=self.rows[i]
            p=r[0] if isinstance(r,(tuple,list)) else r
            return tf(Image.open(p).convert('RGB'))
    dl=DataLoader(DS(rows), batch_size=32, shuffle=False, num_workers=2)
    out=[]
    model.eval()
    with torch.no_grad():
        for x in dl:
            out.append(model(x.to(device)).cpu().numpy())
    return np.concatenate(out,0)

np.save(OUT_ROOT/'stage2_logits_known_val.npy', logits_for(split['known_val']))
np.save(OUT_ROOT/'stage2_logits_known_test.npy', logits_for(split['known_test']))
np.save(OUT_ROOT/'stage2_logits_unknown_test.npy', logits_for(split['unknown_test']))
print('Saved stage2 logits')


In [ ]:
# 10) Integrated two-stage metrics
import json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

split = json.loads((OUT_ROOT/'split_manifest.json').read_text())
class_to_idx = {c:i for i,c in enumerate(split['known_classes'])}

def margin(logits):
    p=np.partition(logits, -2, axis=1)
    return p[:,-1]-p[:,-2]

s_nva=np.load(OUT_ROOT/'stage1_scores_normal_val.npy')
s_nts=np.load(OUT_ROOT/'stage1_scores_normal_test.npy')
s_kts=np.load(OUT_ROOT/'stage1_scores_known_test.npy')
s_uts=np.load(OUT_ROOT/'stage1_scores_unknown_test.npy')

lv=np.load(OUT_ROOT/'stage2_logits_known_val.npy')
lk=np.load(OUT_ROOT/'stage2_logits_known_test.npy')
lu=np.load(OUT_ROOT/'stage2_logits_unknown_test.npy')

m_val=margin(lv); m_k=margin(lk); m_u=margin(lu)
pk=lk.argmax(1)
yk=np.array([class_to_idx[c] for _,c in split['known_test']])

auroc=float(roc_auc_score(np.r_[np.zeros(len(s_nts)), np.ones(len(s_kts)+len(s_uts))], np.r_[s_nts, s_kts, s_uts]))
rows=[]
for fpr in [0.05,0.10,0.20]:
    tau=float(np.quantile(s_nva, 1-fpr))
    kappa=float(np.quantile(m_val, fpr))
    n_def=s_nts>tau; k_def=s_kts>tau; u_def=s_uts>tau
    k_unk=m_k<kappa; u_unk=m_u<kappa
    k_succ=k_def & (~k_unk) & (pk==yk)
    u_succ=u_def & u_unk
    rows.append({
        'fpr_target':fpr,
        'AUROC_defect_screening':auroc,
        'TPR_defect@FPR':float(np.mean(np.r_[k_def,u_def])),
        'TPR_unknown@FPR':float(np.mean(u_def)),
        'FPR_known_as_def@FPR':float(np.mean(k_def)),
        'FPR_normal_realized':float(np.mean(n_def)),
        'two_stage_known_success':float(np.mean(k_succ)),
        'two_stage_unknown_success':float(np.mean(u_succ)),
        'stage2_unknown_rate_known_all':float(np.mean(k_unk)),
        'stage2_unknown_rate_unknown_all':float(np.mean(u_unk)),
    })

res=pd.DataFrame(rows)
res.to_csv(OUT_ROOT/'cflow_two_stage_summary.csv', index=False)
print('Saved:', OUT_ROOT/'cflow_two_stage_summary.csv')
display(res)


In [ ]:
# 11) Plot summary
import pandas as pd, matplotlib.pyplot as plt
res=pd.read_csv(OUT_ROOT/'cflow_two_stage_summary.csv')
plt.figure(figsize=(7,4))
plt.plot(res['fpr_target'], res['TPR_defect@FPR'], marker='o', label='TPR_defect')
plt.plot(res['fpr_target'], res['TPR_unknown@FPR'], marker='o', label='TPR_unknown')
plt.plot(res['fpr_target'], res['FPR_normal_realized'], marker='o', label='FPR_normal')
plt.ylim(0,1); plt.legend(); plt.xlabel('FPR target'); plt.ylabel('rate')
plt.tight_layout()
plt.savefig(OUT_ROOT/'plot_cflow_two_stage_rates.png', dpi=140)
plt.show()
print('Saved:', OUT_ROOT/'plot_cflow_two_stage_rates.png')


## Rerun Notes

1. If NumPy changed, rerun cell 0, restart runtime, then run from cell 1.
2. If CFLOW output parsing fails in cell 8, inspect `/content/cflow-ad` for score artifacts and adjust parser once.
3. Main outputs are in `/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/`.
